In [1]:
cfg = dict(
    seq_length  = 160,
    d_model     = 256,
    latent_dim  = 64,   # latent dimension
    n_head      = 8,
    enc_layers  = 7,
    dec_layers  = 7,
    ff_dim      = 1024,
    dropout     = 0.05,
    emb_dropout = 0.05,

    # special token indices (match your vocabulary)
    pad_idx     = 0,
    sos_idx     = 2,
    eos_idx     = 3,
    # -------- regularization tweaks --------
    label_smoothing = 0.02,   # 0.0 to disable
    corruption_p     = 0.05,  # word-dropout on decoder inputs (train only)

    # -------- validation / decoding --------
    beam_every  = 5,   # run beam metrics every N epochs
    beam_size   = 5)

In [2]:
import torch, torch.nn as nn
import model_bs as mdl
import data_utils as du

# --- paths/config you already have ---
vocab_path   = "/Users/md_halim_mondol/Desktop/LTVAE/vocab.json"
ckpt_path    = "/Users/md_halim_mondol/Desktop/LTVAE/checkpoints/best_model.pth"
Dye_csv     = "/Users/md_halim_mondol/Desktop/LTVAE/Data/dye.csv"
test_csv     = "/Users/md_halim_mondol/Desktop/LTVAE/Data/Test.csv"
Test_pubchem = "/Users/md_halim_mondol/Desktop/LTVAE/Data/Test_pubchem.csv"

# --- load vocab ---
token_to_idx, idx_to_token = du.load_or_create_vocabulary(csv_paths=[], cache_path=vocab_path, test_smiles=None)
assert token_to_idx["<PAD>"] == cfg["pad_idx"]
assert token_to_idx["<SOS>"] == cfg["sos_idx"]
assert token_to_idx["<EOS>"] == cfg["eos_idx"]

# --- build the same architecture you trained ---
model = mdl.LSTM_VAE_Trans(
    vocab_size=len(token_to_idx),
    d_model=cfg["d_model"],
    latent_dim=cfg["latent_dim"],
    pad_idx=cfg["pad_idx"],
    sos_idx=cfg["sos_idx"],
    eos_idx=cfg["eos_idx"],
    enc_layers=cfg["enc_layers"],
    dec_layers=cfg["dec_layers"],
    nhead=cfg["n_head"],
    dropout=cfg["dropout"],
    max_len=cfg["seq_length"],
    dim_feedforward=cfg["ff_dim"])

# --- load weights robustly (handles 'module.' prefixes if any) ---
state = torch.load(ckpt_path, map_location="cpu")
try:
    model.load_state_dict(state, strict=True)
except RuntimeError:
    # remove a leading 'module.' if the checkpoint came from DataParallel
    from collections import OrderedDict
    new_state = OrderedDict()
    for k, v in state.items():
        new_state[k.replace("module.", "", 1)] = v
    model.load_state_dict(new_state, strict=True)

# --- device & optional DataParallel for speed (not required) ---
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device)
print("Trainable params:", du.count_parameters(model))
print(f"Encoder parameters: {du.count_parameters(model.encoder)}")
model.eval()

# If you want to keep everything single-GPU-friendly for beam_search, DON'T wrap in DataParallel.
# If you DO wrap, remember to pass model.module to functions that call custom methods.

[vocab] loaded cached vocabulary from /Users/md_halim_mondol/Desktop/LTVAE/vocab.json (69 tokens)
Trainable params: 10246597
Encoder parameters: 2786048


LSTM_VAE_Trans(
  (encoder): EncoderBiLSTM(
    (emb): Embedding(69, 256, padding_idx=0)
    (emb_ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (emb_do): Dropout(p=0.1, inplace=False)
    (lstm): LSTM(256, 128, num_layers=7, batch_first=True, dropout=0.05, bidirectional=True)
    (out_do): Dropout(p=0.05, inplace=False)
    (seq_ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (pool_ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (to_mu): Linear(in_features=256, out_features=64, bias=True)
  (to_logvar): Linear(in_features=256, out_features=64, bias=True)
  (latent_to_token): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (emb): Embedding(69, 256, padding_idx=0)
    (emb_ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (pe): PositionalEncoding(
      (dropout): Dropout(p=0.05, inplace=False)
    )

In [3]:
import pandas as pd
from inference import reconstruct_smiles_table
import metrics as met

# Run a 4k sample, but reconstruct in small batches to avoid memory errors.
SAMPLE_N = 4000
BATCH_SIZE = 32
test_smiles = pd.read_csv(Test_pubchem, nrows=SAMPLE_N)["smiles"].dropna().astype(str).tolist()
print(f"Running reconstruction on {len(test_smiles)} PubChem SMILES with batch_size={BATCH_SIZE}")

# Use the *unwrapped* model object for beam_search
m = model  # (if you ever wrap with DataParallel, use: model.module)

df_rec = reconstruct_smiles_table( smiles_list=test_smiles, test_csv=None, model=m, token_to_idx=token_to_idx, idx_to_token=idx_to_token, seq_length=cfg["seq_length"], pad_idx=cfg["pad_idx"], sos_idx=cfg["sos_idx"], eos_idx=cfg["eos_idx"], device=device, mode="beam", beam_size=cfg["beam_size"], batch_size=BATCH_SIZE, progress_every=25)
df_rec.to_csv("Data/Test_pubchem_reconstruction_beam_4k.csv", index=False)

# show a preview
display(df_rec.head(10))

# ------------------------------------------------------------------
# 1.  Token-level accuracy (micro-average over SMILES tokens)
# ------------------------------------------------------------------
def token_accuracy_row(gold, pred):
    g = du.tokenize_smiles(gold)
    p = du.tokenize_smiles(pred)
    L = min(len(g), len(p))
    if L == 0:
        return 0, 0
    correct = sum(gi == pi for gi, pi in zip(g[:L], p[:L]))
    total   = L
    return correct, total

tot_corr = tot_tok = 0
for g, p in zip(df_rec["input"], df_rec["reconstructed"]):
    c, t = token_accuracy_row(g, p)
    tot_corr += c
    tot_tok  += t

beam_token_acc = tot_corr / tot_tok if tot_tok else 0.0
print(f"Token level test accuracy (beam): {beam_token_acc:.4f}")

# ------------------------------------------------------------------
# 2.  Sequence-level (exact-match) accuracy
# ------------------------------------------------------------------
exact_match_acc = (df_rec["input"] == df_rec["reconstructed"]).mean()
print(f"Exact SMILES match accuracy (beam): {exact_match_acc:.4f}")


# ---- summary metrics ----
valid_ratio = (df_rec["valid"] == "yes").mean() if len(df_rec) else float("nan")
avg_lev     = df_rec["lev"].mean() if len(df_rec) else float("nan")

print(f"[beam] validity ratio: {valid_ratio:.3f}")
print(f"[beam] average Levenshtein: {avg_lev:.3f}")

Running reconstruction on 4000 PubChem SMILES with batch_size=32
decoded batch 1/125 (32/4000 SMILES)
decoded batch 25/125 (800/4000 SMILES)
decoded batch 50/125 (1600/4000 SMILES)
decoded batch 75/125 (2400/4000 SMILES)
decoded batch 100/125 (3200/4000 SMILES)
decoded batch 125/125 (4000/4000 SMILES)


,input,reconstructed,valid,lev
0,CC(=O)c1c(C)c2cnc(Nc3ccc(N4CCNCC4)cn3)nc2n(C2C...,CC(=O)c1c(C)c2cnc(Nc3ccc(N4CCNCC4)cn3)nc2n(C2C...,yes,0
1,COC1C(N(C)C(=O)c2ccccc2)CC2OC1(C)n1c3ccccc3c3c...,COC1CN(C(=O)Cc2ccccc2)CC(C)=C1c1oc2c3cccc4c5c(...,no,30
2,NC(=O)c1cccc2cn(-c3ccc(C4CCCNC4)cc3)nc12,NC(=O)c1cccc2cn(-c3ccc(C4CCCNC4)cc3)nc12,yes,0
3,Cc1cc(CC(NC(=O)N2CCC(c3cc4ccccc4[nH]c3=O)CC2)C...,Cc1cc(CC(NC2CCN(C(=O)c3oc4ccccc4c3=O)CC2)=O)CN...,no,35
4,CC(Oc1c(Cl)cccc1Cl)C1=NCCN1,CC(Oc1c(Cl)cccc1Cl)C1=NCCN1,yes,0
5,CC(=O)Nc1ccc(O)cc1,CC(=O)Nc1ccc(O)cc1,yes,0
6,CCCCCCCCCCCCCCCCCCCCCCO,CC1CCCCCCCCCCCCC[NH2+]CCCCCO,no,7
7,N#Cc1ccc2c(c1)N(CCCN1CCC(O)CC1)c1ccccc1S2,N#Cc1ccc2c(c1)N(CCCN1CCC(O)CC1)c1ccccc1S2,yes,0
8,Clc1ccc(COC(Cn2ccnc2)c2ccc(Cl)cc2Cl)c(Cl)c1,Clc1ccc(COC[C@H](Cn2ccnc2)c2ccc(Cl)cc2Cl)c(Cl)c1,yes,5
9,CCCCC(F)(F)C1(O)CCC2C(CC(=O)C2CCCCCCC(=O)O)O1,CCCCC(O)=C1CCC2(CC(=O)CC1CCCCCCC(=O)OC2)O1,no,12


Token level test accuracy (beam): 0.7196
Exact SMILES match accuracy (beam): 0.3387
[beam] validity ratio: 0.672
[beam] average Levenshtein: 8.581
